# Лабораторная работа №10: Эффективный fine-tuning VLM с помощью Unsloth

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Повторить SFT на том же датасете через Unsloth.
- Сравнить LoRA rank и accumulation короткими runs.
- Построить quality-time-memory benchmark.


## Теоретическая часть

Эта работа оценивает не новый task, а efficiency: wall-clock, peak VRAM, throughput и качество при близких hyperparameters. Ускорение без эквивалентного evaluation не является результатом.


### Математическая постановка

Throughput $=N_{tokens}/T$, peak memory берётся из `torch.cuda.max_memory_allocated()`. Полезно строить Pareto frontier по quality, time и memory.


### Материалы

- [Unsloth docs](https://docs.unsloth.ai/)
- [PEFT](https://huggingface.co/docs/peft/index)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практический протокол

Тот же manifest и та же Qwen2.5-VL, что в работе 9. В Colab требуется совместимая пара версий Unsloth и TRL; зафиксируйте версии и аппаратное обеспечение. `FAST_DEV_RUN=True` всё равно запускает обучение, но только для smoke. Отсутствующие данные или GPU приводят к явной ошибке, а не к демонстрационному benchmark.

In [1]:
from pathlib import Path
import json, os, random, time
import numpy as np
SEED = 42
FAST_DEV_RUN = False  # True: проверка механики на малой выборке, НЕ отчётный эксперимент
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
def save_json(name, data):
    (ARTIFACTS/name).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str))
from pathlib import Path
import json, hashlib
from datasets import load_dataset

SCIENCEQA_ID = 'derek-thomas/ScienceQA'
SCIENCEQA_REVISION = 'f18b0a70359ebfb41f658fd564208d0355b013f4'

def prepare_scienceqa(directory, smoke=False):
    """Fixed official splits, image-level deduplication, explicitly defined shifts."""
    directory = Path(directory).resolve(); directory.mkdir(parents=True, exist_ok=True)
    manifest = directory / ('manifest-smoke.jsonl' if smoke else 'manifest.jsonl')
    limits = {'train': 8 if smoke else 512, 'validation': 4 if smoke else 64,
              'id_test': 4 if smoke else 64, 'ood_test': 4 if smoke else 64,
              'general_test': 4 if smoke else 64}
    if manifest.exists():
        records = [json.loads(s) for s in manifest.read_text().splitlines()]
        if all(Path(r['image']).is_file() for r in records) and all(sum(r['split']==s for r in records)==n for s,n in limits.items()):
            return manifest
    records, seen, counts = [], set(), dict.fromkeys(limits, 0)
    for official in ('train', 'validation', 'test'):
        data = load_dataset(SCIENCEQA_ID, revision=SCIENCEQA_REVISION, split=official, streaming=True)
        for index, row in enumerate(data):
            image = row['image']
            if image is None: continue
            if official != 'test':
                if row['subject'] != 'natural science' or int(row['grade'].removeprefix('grade')) > 6: continue
                split = official
            elif row['subject'] != 'natural science': split = 'general_test'
            elif int(row['grade'].removeprefix('grade')) > 6: split = 'ood_test'
            else: split = 'id_test'
            if counts[split] >= limits[split]: continue
            image = image.convert('RGB')
            digest = hashlib.sha256(str(image.size).encode()+image.tobytes()).hexdigest()
            if digest in seen: continue
            seen.add(digest)
            path = directory / f'{digest}.png'; image.save(path)
            answer = int(row['answer']); choices = row['choices']
            question = row['question']+'\n'+'\n'.join(f'{chr(65+i)}. {v}' for i,v in enumerate(choices))+'\nReturn only the letter of the correct option.'
            records.append({'id': f'scienceqa-{official}-{index}', 'split': split,
                            'image': str(path), 'question': question, 'answer': chr(65+answer),
                            'choices': choices, 'answer_index': answer, 'label_type': 'multiple_choice',
                            'license': 'MIT (code license, lupantech/ScienceQA)',
                            'source_url': f'https://huggingface.co/datasets/{SCIENCEQA_ID}/tree/{SCIENCEQA_REVISION}',
                            'subject':row['subject'], 'grade':row['grade'], 'image_sha256':digest})
            counts[split] += 1
            required = [official] if official != 'test' else ['id_test','ood_test','general_test']
            if all(counts[s] == limits[s] for s in required): break
    if counts != limits: raise RuntimeError(f'Insufficient licensed image examples: {counts}; expected {limits}')
    manifest.write_text('\n'.join(json.dumps(r,ensure_ascii=False) for r in records)+'\n')
    (directory/'ATTRIBUTION.md').write_text('ScienceQA — Lu et al., NeurIPS 2022. Dataset distribution: '+SCIENCEQA_ID+'\nRevision: '+SCIENCEQA_REVISION+'\nDataset-card license: CC-BY-SA-4.0, https://creativecommons.org/licenses/by-sa/4.0/\nOriginal project: https://scienceqa.github.io/\nImage subsets and manifests retain attribution; downloaded images are not committed to this course.\nID: natural science grades 1–6; OOD: natural science grades 7–12; general: other subjects. This is an operational shift, not a claim of universal model ability.\n')
    return manifest


## Оценивание

Десять обязательных предметных этапов — по 1 баллу каждый, всего 10 баллов. Отдельное творческое исследование — до 1 дополнительного балла. Каждый этап принимается по указанному наблюдаемому результату.

### Одинаковые данные для сравнения

*Вес: 1 балл.*

**Постановка.** Загрузите ту же фиксированную ScienceQA-подвыборку через prepare_scienceqa, что в работе 09. Для сравнения библиотек нельзя менять наборы, вопросы или разбиения.

**Результат.** Манифест и изображения.

**Критерий принятия.** ID/хеши записей совпадают с протоколом 09; train и test не пересекаются.


In [2]:
import hashlib, re, shutil
from collections import Counter, defaultdict
from PIL import Image

def load_jsonl(path):
    path = Path(path)
    return [json.loads(s) for s in path.read_text().splitlines() if s.strip()] if path.exists() else []

def find_lab9(name):
    """Ищет артефакт работы 09: CV_LAB9_DIR, текущий artifacts/, соседние каталоги lab09/lab9."""
    cands = [Path(os.getenv('CV_LAB9_DIR', '__none__')) / name, ARTIFACTS / name, Path('../lab09/artifacts') / name, Path('../lab9/artifacts') / name]
    return next((c for c in cands if c.exists()), None)

LIMITS = {'train': 8 if FAST_DEV_RUN else 512, 'validation': 4 if FAST_DEV_RUN else 64, 'id_test': 4 if FAST_DEV_RUN else 64,
          'ood_test': 4 if FAST_DEV_RUN else 64, 'general_test': 4 if FAST_DEV_RUN else 64}
manifest_path = Path(prepare_scienceqa(os.getenv('CV_DATA_DIR', 'data/lab9'), smoke=FAST_DEV_RUN))   # тот же загрузчик и ревизия, что в работе 09
records = load_jsonl(manifest_path)
by_split = defaultdict(list)
for r in records: by_split[r['split']].append(r)
grade = lambda r: int(r['grade'].removeprefix('grade'))

assert {s: len(v) for s, v in by_split.items()} == LIMITS, {s: len(v) for s, v in by_split.items()}
assert len({r['id'] for r in records}) == len(records) and len({r['image_sha256'] for r in records}) == len(records)   # ID и изображения уникальны
tr_h = {r['image_sha256'] for r in by_split['train']}; te_h = {r['image_sha256'] for s in ('id_test', 'ood_test', 'general_test') for r in by_split[s]}
tr_i = {r['id'] for r in by_split['train']}; te_i = {r['id'] for s in ('id_test', 'ood_test', 'general_test') for r in by_split[s]}
assert not tr_h & te_h and not tr_i & te_i                                                      # train и test не пересекаются
for r in by_split['train'] + by_split['id_test']: assert r['subject'] == 'natural science' and grade(r) <= 6
for r in by_split['ood_test']: assert r['subject'] == 'natural science' and grade(r) >= 7
assert all(Path(r['image']).is_file() for r in records)

# совпадение с протоколом 09: если манифест работы 09 доступен, ID и хеши обязаны совпасть
fingerprint = hashlib.sha256(json.dumps(sorted((r['id'], r['image_sha256'], r['split']) for r in records)).encode()).hexdigest()
m9 = find_lab9('manifest-smoke.jsonl' if FAST_DEV_RUN else 'manifest.jsonl')
matches09 = None
if m9 is not None and m9.resolve() != manifest_path.resolve():
    matches09 = {(r['id'], r['image_sha256'], r['split']) for r in load_jsonl(m9)} == {(r['id'], r['image_sha256'], r['split']) for r in records}
    assert matches09, 'манифест отличается от работы 09'
save_json('manifest_check.json', {'manifest': str(manifest_path), 'counts': LIMITS, 'dataset': SCIENCEQA_ID, 'revision': SCIENCEQA_REVISION,
                                  'records_fingerprint_sha256': fingerprint, 'lab09_manifest_found': m9 is not None and m9.resolve() != manifest_path.resolve(),
                                  'matches_lab09': matches09, 'train_test_overlap': 0})
shutil.copy(manifest_path, ARTIFACTS / manifest_path.name)
shutil.copy(manifest_path.parent / 'ATTRIBUTION.md', ARTIFACTS / 'ATTRIBUTION.md')
print({s: len(v) for s, v in by_split.items()}, '| fingerprint', fingerprint[:12], '| совпадает с 09:', matches09)

README.md:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

{'train': 512, 'validation': 64, 'ood_test': 64, 'general_test': 64, 'id_test': 64} | fingerprint 397197651b66 | совпадает с 09: None


### Равный вычислительный бюджет

*Вес: 1 балл.*

**Постановка.** Зафиксируйте Qwen2.5-VL-3B-Instruct, GPU, 100 optimizer steps и accumulation 16. Сравнивайте ранги 8 и 16 при одинаковом количестве обработанных примеров.

**Результат.** protocol.json.

**Критерий принятия.** Сохранены GPU, модель, размеры выборок и бюджет; для обеих серий указан один seed.


In [5]:
%pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.4/30.4 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 23.8 MB/s eta 0:00:00


In [6]:
from unsloth import FastVisionModel                       # unsloth импортируется до transformers
from unsloth.trainer import UnslothVisionDataCollator
import torch, importlib.metadata as md
assert torch.cuda.is_available(), 'Нужен GPU (T4 и выше): без него протокол не может быть выполнен, демонстрационный benchmark не создаётся'
GPU = torch.cuda.get_device_name(0)
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
MAX_STEPS = 2 if FAST_DEV_RUN else 100                    # optimizer steps
ACCUM = 2 if FAST_DEV_RUN else 16                         # микробатчей на один optimizer step (micro batch = 1)
LR, MAX_GRAD_NORM, WARMUP_RATIO = 1e-4, 1.0, 0.1
IMAGE_MAX_PIXELS, MAX_NEW_TOKENS = 262144, 8
RUNS = {'r8_a16': {'r': 8, 'alpha': 16, 'dir': ARTIFACTS / 'unsloth_r8_a16'},     # alpha = 2*rank
        'r16_a16': {'r': 16, 'alpha': 32, 'dir': ARTIFACTS / 'unsloth_r16_a16'}}  # имя каталога по заданию, фактический alpha = 32 (записан в training.json)

train_recs = by_split['train']
n_examples = MAX_STEPS * ACCUM
order, rs = [], np.random.RandomState(SEED)
while len(order) < n_examples: order += rs.permutation(len(train_recs)).tolist()
order = order[:n_examples]                                # один и тот же порядок примеров для обоих рангов
order_sha = hashlib.sha256(json.dumps(order).encode()).hexdigest()

def ver(n):
    try: return md.version(n)
    except Exception: return None
protocol = {'gpu': GPU, 'gpu_total_memory_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), 'model': MODEL_ID,
            'quantization': '4-bit NF4 (bitsandbytes, через Unsloth)', 'optimizer_steps': MAX_STEPS, 'accumulation': ACCUM, 'micro_batch': 1,
            'effective_batch': ACCUM, 'examples_processed_per_run': n_examples, 'lr': LR, 'lr_schedule': 'cosine, warmup 10%', 'weight_decay': 0.0,
            'max_grad_norm': MAX_GRAD_NORM, 'seed': SEED, 'seed_note': 'один seed для обеих серий', 'data_order_sha256': order_sha,
            'image_max_pixels': IMAGE_MAX_PIXELS, 'sizes': {s: len(v) for s, v in by_split.items()}, 'manifest_fingerprint': fingerprint,
            'trainable_modules': 'LoRA на attention и MLP языковой части; vision tower заморожен', 'ranks': {k: {'r': v['r'], 'alpha': v['alpha']} for k, v in RUNS.items()},
            'versions': {n: ver(n) for n in ['unsloth', 'unsloth_zoo', 'transformers', 'peft', 'trl', 'bitsandbytes', 'torch']}, 'fast_dev_run': FAST_DEV_RUN}
save_json('protocol.json', protocol)
print(json.dumps({k: v for k, v in protocol.items() if k != 'versions'}, ensure_ascii=False, indent=1)[:1500])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
{
 "gpu": "Tesla T4",
 "gpu_total_memory_gb": 14.56,
 "model": "Qwen/Qwen2.5-VL-3B-Instruct",
 "quantization": "4-bit NF4 (bitsandbytes, через Unsloth)",
 "optimizer_steps": 100,
 "accumulation": 16,
 "micro_batch": 1,
 "effective_batch": 16,
 "examples_processed_per_run": 1600,
 "lr": 0.0001,
 "lr_schedule": "cosine, warmup 10%",
 "weight_decay": 0.0,
 "max_grad_norm": 1.0,
 "seed": 42,
 "seed_note": "один seed для обеих серий",
 "data_order_sha256": "bed62d6f61dce79d4a508c21df4752c5a83c0671ffc0343aff1cc9a1ef7ac4c9",
 "image_max_pixels": 262144,
 "size

### Цикл мультимодального SFT

*Вес: 1 балл.*

**Постановка.** Реализуйте train_adapter с UnslothVisionDataCollator, AdamW, накоплением градиентов и ограничением их нормы. Измеряйте время с CUDA synchronize, сохраняйте историю loss и обученный адаптер.

**Результат.** Функция train_adapter.

**Критерий принятия.** На каждый optimizer step приходится accumulation микробатчей; измерения реальны; нечисловой loss вызывает ошибку.


In [7]:
import gc, math
from transformers import get_cosine_schedule_with_warmup

def limit_pixels(img, max_pixels=IMAGE_MAX_PIXELS):
    w, h = img.size
    if w * h <= max_pixels: return img
    s = (max_pixels / (w * h)) ** 0.5
    return img.resize((max(28, int(w * s)), max(28, int(h * s))), Image.BICUBIC)

def to_conversation(rec):
    return {'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': limit_pixels(Image.open(rec['image']).convert('RGB'))},
                                                     {'type': 'text', 'text': rec['question']}]},
                         {'role': 'assistant', 'content': [{'type': 'text', 'text': rec['answer']}]}]}

def last_subseq_end(seq, sub):
    """Индекс сразу после ПОСЛЕДНЕГО вхождения подпоследовательности sub в seq (или -1)."""
    for i in range(len(seq) - len(sub), -1, -1):
        if seq[i:i + len(sub)] == sub: return i + len(sub)
    return -1
assert last_subseq_end([1, 2, 3, 2, 3, 9], [2, 3]) == 5 and last_subseq_end([1, 2], [3]) == -1

def mask_prompt_labels(batch, marker_ids):
    """Как в работе 09 (LLaMA-Factory): loss считается только по ответу ассистента; prompt и изображение маскируются."""
    labels = batch['labels'].clone()
    for i in range(labels.shape[0]):
        end = last_subseq_end(batch['input_ids'][i].tolist(), marker_ids)
        if end < 0: raise ValueError('не найден маркер начала ответа ассистента')
        labels[i, :end] = -100
    if not (labels != -100).any(): raise ValueError('после маскирования нет токенов ответа')
    return labels

def train_adapter(r, alpha, out_dir, order=order, steps=MAX_STEPS, accum=ACCUM, lr=LR):
    """Мультимодальный SFT: UnslothVisionDataCollator, AdamW, накопление градиентов (accum микробатчей на шаг), clip по норме,
    честное время (cuda synchronize), пиковая память, история loss. Каждый запуск начинается с ИСХОДНОЙ модели."""
    out_dir = Path(out_dir); (out_dir / 'adapter').mkdir(parents=True, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model, tokenizer = FastVisionModel.from_pretrained(MODEL_ID, load_in_4bit=True, use_gradient_checkpointing='unsloth')
    model = FastVisionModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True, finetune_attention_modules=True,
                                           finetune_mlp_modules=True, r=r, lora_alpha=alpha, lora_dropout=0.0, bias='none', random_state=SEED)
    FastVisionModel.for_training(model)
    collator = UnslothVisionDataCollator(model, tokenizer)
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = int(sum(p.numel() for p in params)); assert params
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(WARMUP_RATIO * steps), steps)
    use_bf16 = torch.cuda.is_bf16_supported(); amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=not use_bf16)                 # T4: fp16 + loss scaling
    marker = getattr(tokenizer, 'tokenizer', tokenizer).encode('<|im_start|>assistant\n', add_special_tokens=False)
    history, micro_count, tokens_total, skipped = [], 0, 0, 0
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t_start = time.perf_counter()
    for step in range(steps):
        torch.cuda.synchronize(); t0 = time.perf_counter(); opt.zero_grad(set_to_none=True); loss_sum, step_tokens = 0.0, 0
        for m in range(accum):
            rec = train_recs[order[step * accum + m]]
            batch = collator([to_conversation(rec)])
            labels = mask_prompt_labels(batch, marker).to('cuda')
            inputs = {k: v.to('cuda') for k, v in batch.items() if k != 'labels'}
            with torch.autocast('cuda', dtype=amp_dtype):
                loss = model(**inputs, labels=labels).loss
            if not torch.isfinite(loss): raise FloatingPointError(f'нечисловой loss на шаге {step}, микробатче {m}: {loss.item()}')
            scaler.scale(loss / accum).backward()
            loss_sum += float(loss.item()); step_tokens += int(inputs['attention_mask'].sum().item()); micro_count += 1
        scaler.unscale_(opt); gnorm = float(torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM))
        if not math.isfinite(gnorm):
            if use_bf16: raise FloatingPointError(f'нечисловая норма градиента на шаге {step}')
            skipped += 1                                                          # GradScaler пропустит шаг при inf (fp16)
        lr_now = opt.param_groups[0]['lr']; scaler.step(opt); scaler.update(); sched.step()
        torch.cuda.synchronize(); dt = time.perf_counter() - t0; tokens_total += step_tokens
        history.append({'step': step + 1, 'loss': loss_sum / accum, 'grad_norm': gnorm, 'lr': lr_now, 'seconds': dt, 'tokens': step_tokens})
    torch.cuda.synchronize(); total = time.perf_counter() - t_start
    assert micro_count == steps * accum and len(history) == steps and all(math.isfinite(h['loss']) for h in history)
    peak = torch.cuda.max_memory_allocated() / 2**30
    model.save_pretrained(str(out_dir / 'adapter')); tokenizer.save_pretrained(str(out_dir / 'adapter'))
    result = {'r': r, 'alpha': alpha, 'optimizer_steps': steps, 'accumulation': accum, 'microbatches': micro_count, 'examples_processed': micro_count,
              'n_trainable_params': n_trainable, 'total_seconds': total, 'peak_cuda_memory_gb': peak, 'tokens_total': tokens_total,
              'tokens_per_second': tokens_total / total, 'skipped_steps_fp16': skipped, 'precision': 'bf16' if use_bf16 else 'fp16+GradScaler',
              'data_order_sha256': order_sha, 'seed': SEED, 'gpu': GPU, 'history': history, 'adapter_dir': str(out_dir / 'adapter'), 'fast_dev_run': FAST_DEV_RUN}
    (out_dir / 'training.json').write_text(json.dumps(result, ensure_ascii=False, indent=1))
    del model, tokenizer, opt, sched, collator, params; gc.collect(); torch.cuda.empty_cache()
    return result

def run_rank(tag):
    cfg = RUNS[tag]; tj = cfg['dir'] / 'training.json'
    if tj.exists() and (cfg['dir'] / 'adapter' / 'adapter_model.safetensors').exists():          # возобновление на уровне прогона
        old = json.loads(tj.read_text())
        if old['optimizer_steps'] == MAX_STEPS and old['accumulation'] == ACCUM and old['data_order_sha256'] == order_sha and (old['r'], old['alpha']) == (cfg['r'], cfg['alpha']):
            print(tag, ': готовый прогон найден, обучение пропущено'); return old
    res = train_adapter(cfg['r'], cfg['alpha'], cfg['dir'])
    return res

def check_run(res, tag):
    cfg = RUNS[tag]
    assert res['optimizer_steps'] == MAX_STEPS and res['microbatches'] == MAX_STEPS * ACCUM, 'неполный бюджет'
    assert (res['r'], res['alpha']) == (cfg['r'], cfg['alpha']) and res['data_order_sha256'] == order_sha
    assert all(math.isfinite(h['loss']) for h in res['history']) and res['peak_cuda_memory_gb'] > 0 and res['total_seconds'] > 0
    assert Path(res['adapter_dir'], 'adapter_model.safetensors').is_file()
print('train_adapter определена; FAST_DEV_RUN =', FAST_DEV_RUN, '| микробатчей на шаг =', ACCUM)

train_adapter определена; FAST_DEV_RUN = False | микробатчей на шаг = 16


### LoRA ранга 8

*Вес: 1 балл.*

**Постановка.** Создайте FastVisionModel и LoRA rank 8, alpha 16. Обучите модель по общему протоколу и сохраните адаптер отдельно.

**Результат.** unsloth_r8_a16/adapter и training.json.

**Критерий принятия.** Фактически выполнено 100 optimizer steps; зафиксированы loss, время и пиковая CUDA-память.


In [8]:
res_r8 = run_rank('r8_a16'); check_run(res_r8, 'r8_a16')
print(f"r=8, alpha=16: шагов={res_r8['optimizer_steps']}, микробатчей={res_r8['microbatches']}, время={res_r8['total_seconds']:.0f}s, "
      f"пик памяти={res_r8['peak_cuda_memory_gb']:.2f} ГБ, loss {res_r8['history'][0]['loss']:.3f} -> {res_r8['history'][-1]['loss']:.3f}, "
      f"обучаемых параметров={res_r8['n_trainable_params']:,}")

==((====))==  Unsloth 2026.10.6: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found
Unsloth: compiling each transformer block as one region (Qwen2_5_VLDecoderLayer).
Unsloth: Model does not

Unsloth: Restored added_tokens_decoder metadata in artifacts/unsloth_r8_a16/adapter/tokenizer_config.json.


r=8, alpha=16: шагов=100, микробатчей=1600, время=1024s, пик памяти=4.06 ГБ, loss 0.273 -> 0.010, обучаемых параметров=14,966,784


### LoRA ранга 16

*Вес: 1 балл.*

**Постановка.** Повторите эксперимент с rank 16, alpha 32. Начинайте с исходной модели, а не с адаптера ранга 8.

**Результат.** unsloth_r16_a16/adapter и training.json.

**Критерий принятия.** Меняется только ранг и соответствующий alpha; порядок данных, число шагов и effective batch совпадают.


In [9]:
res_r16 = run_rank('r16_a16'); check_run(res_r16, 'r16_a16')           # старт с исходной модели; меняются только rank и alpha
assert res_r16['data_order_sha256'] == res_r8['data_order_sha256'] and res_r16['accumulation'] == res_r8['accumulation'] and res_r16['seed'] == res_r8['seed']
print(f"r=16, alpha=32: шагов={res_r16['optimizer_steps']}, микробатчей={res_r16['microbatches']}, время={res_r16['total_seconds']:.0f}s, "
      f"пик памяти={res_r16['peak_cuda_memory_gb']:.2f} ГБ, loss {res_r16['history'][0]['loss']:.3f} -> {res_r16['history'][-1]['loss']:.3f}, "
      f"обучаемых параметров={res_r16['n_trainable_params']:,}")

==((====))==  Unsloth 2026.10.6: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found
Unsloth: Model does not have a default image size - using 512


Unsloth: Restored added_tokens_decoder metadata in artifacts/unsloth_r16_a16/adapter/tokenizer_config.json.


r=16, alpha=32: шагов=100, микробатчей=1600, время=979s, пик памяти=4.28 ГБ, loss 0.273 -> 0.006, обучаемых параметров=29,933,568


### Траектории оптимизации

*Вес: 1 балл.*

**Постановка.** Постройте график loss по optimizer steps для двух рангов. Сопоставьте число шагов, время и пиковую память, прежде чем делать вывод о полезности большего ранга.

**Результат.** loss_comparison.png и benchmark.json.

**Критерий принятия.** График построен по реальным историям обучения, оси и ранги подписаны; отсутствующие измерения не заменены константой.


In [10]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
runs = {'r8_a16': res_r8, 'r16_a16': res_r16}
def summ(res):
    L = [h['loss'] for h in res['history']]; k = min(10, len(L))
    return {'rank': res['r'], 'alpha': res['alpha'], 'optimizer_steps': res['optimizer_steps'], 'microbatches': res['microbatches'],
            'total_seconds': res['total_seconds'], 'mean_step_seconds': res['total_seconds'] / res['optimizer_steps'],
            'peak_cuda_memory_gb': res['peak_cuda_memory_gb'], 'tokens_total': res['tokens_total'], 'tokens_per_second': res['tokens_per_second'],
            'n_trainable_params': res['n_trainable_params'], 'loss_first': float(np.mean(L[:k])), 'loss_last': float(np.mean(L[-k:])), 'precision': res['precision']}
bench = {tag: summ(r) for tag, r in runs.items()}
a, b = bench['r8_a16'], bench['r16_a16']
assert a['optimizer_steps'] == b['optimizer_steps'] and a['microbatches'] == b['microbatches']        # сопоставимый бюджет
bench['ratio_r16_over_r8'] = {'time': b['total_seconds'] / a['total_seconds'], 'peak_memory': b['peak_cuda_memory_gb'] / a['peak_cuda_memory_gb'],
                              'trainable_params': b['n_trainable_params'] / a['n_trainable_params'], 'delta_loss_last': b['loss_last'] - a['loss_last']}
bench['gpu'] = GPU; bench['note'] = 'loss - обучающий; без оценки на отложенных данных нельзя судить о пользе большего ранга'
save_json('benchmark.json', bench)

fig, ax = plt.subplots(figsize=(7, 4)); w = max(1, MAX_STEPS // 10)
for tag, res in runs.items():
    s = [h['step'] for h in res['history']]; L = np.array([h['loss'] for h in res['history']])
    ax.plot(s, L, alpha=.3); sm = np.convolve(L, np.ones(w) / w, mode='valid') if len(L) >= w else L
    ax.plot(s[w - 1:] if len(L) >= w else s, sm, label=f"rank {res['r']}, alpha {res['alpha']}")
ax.set_xlabel('optimizer step'); ax.set_ylabel('train loss (ответ ассистента)'); ax.grid(alpha=.3); ax.legend(); ax.set_title('Траектории loss: rank 8 vs rank 16')
plt.tight_layout(); plt.savefig(ARTIFACTS / 'loss_comparison.png', dpi=130); plt.show()
print(f"{'':10s}{'время,с':>10s}{'пик,ГБ':>9s}{'ток/с':>9s}{'params':>12s}{'loss(посл.10)':>15s}")
for tag in runs: x = bench[tag]; print(f"{tag:10s}{x['total_seconds']:10.0f}{x['peak_cuda_memory_gb']:9.2f}{x['tokens_per_second']:9.0f}{x['n_trainable_params']:12,}{x['loss_last']:15.4f}")
print({k: round(v, 3) for k, v in bench['ratio_r16_over_r8'].items()})

             время,с   пик,ГБ    ток/с      params  loss(посл.10)
r8_a16          1024     4.06      223  14,966,784         0.0437
r16_a16          979     4.28      233  29,933,568         0.0326
{'time': 0.957, 'peak_memory': 1.055, 'trainable_params': 2.0, 'delta_loss_last': -0.011}


### Общий интерфейс инференса

*Вес: 1 балл.*

**Постановка.** Реализуйте один greedy-инференс для обоих адаптеров: общие processor, размер изображения, max_new_tokens и извлечение продолжения.

**Результат.** Функция infer.

**Критерий принятия.** Выход не содержит prompt; ответы получаются от активного адаптера на реальном изображении.


In [11]:
def build_messages(question):
    return [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': question}]}]

def load_for_inference(adapter_dir):
    """Загружает базовую модель вместе с адаптером (Unsloth читает base model из adapter_config); после вызова адаптер активен."""
    model, tokenizer = FastVisionModel.from_pretrained(str(adapter_dir), load_in_4bit=True)
    FastVisionModel.for_inference(model)
    return model, tokenizer

def infer(model, tokenizer, image_path, question, max_new_tokens=MAX_NEW_TOKENS):
    """Общий greedy-инференс: один способ подготовки изображения, один prompt, одно число токенов; возвращается только продолжение."""
    image = limit_pixels(Image.open(image_path).convert('RGB'))
    text = tokenizer.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(images=image, text=text, add_special_tokens=False, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tokenizer.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

# проверки без GPU: prompt собирается одинаково и содержит маркер изображения
_t = 'Which is bigger?\nA. x\nB. y\nReturn only the letter of the correct option.'
assert build_messages(_t)[0]['content'][0] == {'type': 'image'} and build_messages(_t)[0]['content'][1]['text'] == _t
_big = Image.new('RGB', (2000, 2000)); assert limit_pixels(_big).size[0] * limit_pixels(_big).size[1] <= IMAGE_MAX_PIXELS
print('infer определена: max_new_tokens =', MAX_NEW_TOKENS, '| greedy (do_sample=False)')

infer определена: max_new_tokens = 8 | greedy (do_sample=False)


### Метрика и ошибки формата

*Вес: 1 балл.*

**Постановка.** Определите exact match правильной буквы и отдельную долю ответов допустимого формата A–Z. Реализуйте подсчёт по split с явными знаменателями; пустая выборка должна вызывать ошибку.

**Результат.** Функция evaluate_answers.

**Критерий принятия.** Верный ответ даёт EM=1, неверный и лишний текст — EM=0; n равен числу всех примеров split.


In [12]:
FORMAT_RE = re.compile(r'^[A-Z]\.?$')

def evaluate_answers(rows, split=None):
    """EM = ответ ровно одна буква A-Z (допустима точка) и она равна эталону; format_rate = доля ответов допустимого формата.
    Знаменатель - все примеры split; пустая выборка и повторяющиеся ID - ошибка."""
    rows = [r for r in rows if split is None or r['split'] == split]
    if not rows: raise ValueError(f'пустая выборка (split={split})')
    if len({r['id'] for r in rows}) != len(rows): raise ValueError('повторяющиеся ID')
    per = []
    for r in rows:
        resp = r['response'].strip(); fmt = bool(FORMAT_RE.fullmatch(resp)); pred = resp.rstrip('.') if fmt else None
        per.append({'id': r['id'], 'gold': r['gold'], 'response': r['response'], 'pred': pred, 'format_ok': fmt, 'correct': int(fmt and pred == r['gold'])})
    n = len(per)
    return {'split': split, 'n': n, 'n_correct': sum(p['correct'] for p in per), 'exact_match': sum(p['correct'] for p in per) / n,
            'n_format_ok': sum(p['format_ok'] for p in per), 'format_rate': sum(p['format_ok'] for p in per) / n, 'per_example': per}

_t = lambda resp, gold='A', i='x': {'id': i, 'split': 's', 'gold': gold, 'response': resp}
assert evaluate_answers([_t('A')])['exact_match'] == 1.0                                   # верный ответ
assert evaluate_answers([_t('B')])['exact_match'] == 0.0                                   # неверный
e = evaluate_answers([_t('A. because it is bigger')]); assert e['exact_match'] == 0.0 and e['format_rate'] == 0.0   # лишний текст
assert evaluate_answers([_t('a')])['format_rate'] == 0.0 and evaluate_answers([_t('')])['exact_match'] == 0.0
e = evaluate_answers([_t('A', i='1'), _t('B', i='2'), _t('The answer is A', i='3'), _t('A', i='4')]); assert e['n'] == 4 and e['exact_match'] == .5 and e['format_rate'] == .75
for bad in ([], [_t('A', i='1'), _t('A', i='1')]):
    try: evaluate_answers(bad); raise AssertionError('ожидалась ошибка')
    except ValueError: pass
print('evaluate_answers: самопроверки пройдены')

evaluate_answers: самопроверки пройдены


### Качество двух адаптеров

*Вес: 1 балл.*

**Постановка.** Примените оба адаптера к одним ID/OOD изображениям и рассчитайте согласованные метрики. Сохраните исходный текст каждого ответа.

**Результат.** predictions_r8_a16.jsonl и predictions_r16_a16.jsonl.

**Критерий принятия.** Наборы ID совпадают между сериями; отчёт содержит exact match, format rate и n на ID/OOD.


In [13]:
ADAPTERS = {tag: cfg['dir'] / 'adapter' for tag, cfg in RUNS.items()}
EVAL_SPLITS = ('id_test', 'ood_test')
eval_recs = [r for s in EVAL_SPLITS for r in by_split[s]]

def predict_all(tag, recs, path, sanity=True):
    """Возобновляемый инференс: ответы активного адаптера пишутся в jsonl по ID."""
    path = Path(path); done = {r['id'] for r in load_jsonl(path)}; todo = [r for r in recs if r['id'] not in done]
    if not todo: return
    model, tok = load_for_inference(ADAPTERS[tag])
    if sanity:
        ex = by_split['validation'][0]; a1, a2 = infer(model, tok, ex['image'], ex['question']), infer(model, tok, ex['image'], ex['question'])
        assert a1 == a2 and ex['question'].split('\n')[0] not in a1, 'ответ недетерминирован или содержит prompt'
    with path.open('a') as f:
        for k, r in enumerate(todo, 1):
            f.write(json.dumps({'id': r['id'], 'split': r['split'], 'gold': r['answer'], 'response': infer(model, tok, r['image'], r['question']), 'adapter': tag},
                               ensure_ascii=False) + '\n'); f.flush()
    del model, tok; gc.collect(); torch.cuda.empty_cache()

pred_rows = {}
for tag in RUNS:
    path = ARTIFACTS / f'predictions_{tag}.jsonl'; predict_all(tag, eval_recs, path); pred_rows[tag] = load_jsonl(path)
ids = {tag: {r['id'] for r in rows} for tag, rows in pred_rows.items()}
assert ids['r8_a16'] == ids['r16_a16'] == {r['id'] for r in eval_recs}                    # наборы ID совпадают

quality = {}
for tag, rows in pred_rows.items():
    quality[tag] = {}
    for s in EVAL_SPLITS:
        e = evaluate_answers(rows, s); assert e['n'] == LIMITS[s]
        quality[tag][s] = {k: e[k] for k in ('n', 'n_correct', 'exact_match', 'n_format_ok', 'format_rate')}
# парное сравнение рангов на одних и тех же изображениях
paired = {}
for s in EVAL_SPLITS:
    c = {tag: {p['id']: p['correct'] for p in evaluate_answers(rows, s)['per_example']} for tag, rows in pred_rows.items()}
    keys = sorted(c['r8_a16']); d = np.array([c['r16_a16'][k] - c['r8_a16'][k] for k in keys]); br = np.random.RandomState(SEED)
    boots = [d[br.randint(0, len(d), len(d))].mean() for _ in range(2000)]
    paired[s] = {'delta_em_r16_minus_r8': float(d.mean()), 'bootstrap_95ci': [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))],
                 'r8_wrong_r16_right': int((d == 1).sum()), 'r8_right_r16_wrong': int((d == -1).sum()), 'n': len(d)}
save_json('quality.json', {'metric': 'EM: ответ ровно одна буква A-Z и равна эталону; format_rate: доля ответов вида A-Z', 'quality': quality, 'paired_r16_vs_r8': paired,
                           'fast_dev_run': FAST_DEV_RUN})
for tag in quality:
    for s in EVAL_SPLITS: q = quality[tag][s]; print(f"{tag:8s} {s:8s} n={q['n']:3d}  EM={q['exact_match']:.3f}  format={q['format_rate']:.3f}")
print(json.dumps(paired, ensure_ascii=False))

==((====))==  Unsloth 2026.10.6: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

==((====))==  Unsloth 2026.10.6: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

r8_a16   id_test  n= 64  EM=0.922  format=1.000
r8_a16   ood_test n= 64  EM=0.734  format=1.000
r16_a16  id_test  n= 64  EM=0.906  format=1.000
r16_a16  ood_test n= 64  EM=0.734  format=1.000
{"id_test": {"delta_em_r16_minus_r8": -0.015625, "bootstrap_95ci": [-0.046875, 0.0], "r8_wrong_r16_right": 0, "r8_right_r16_wrong": 1, "n": 64}, "ood_test": {"delta_em_r16_minus_r8": 0.0, "bootstrap_95ci": [-0.046875, 0.046875], "r8_wrong_r16_right": 1, "r8_right_r16_wrong": 1, "n": 64}}


### Сравнение подходов

*Вес: 1 балл.*

**Постановка.** Сопоставьте качество двух рангов и результат LLaMA-Factory из работы 09. Скорость библиотек сравнивайте только при одинаковых GPU, шагах, effective batch и обучаемых модулях. Отсутствующий baseline обозначьте явно.

**Результат.** comparison.json и вывод.

**Критерий принятия.** Не заявлен speedup по несопоставимым измерениям; отсутствие baseline не выдано за нулевое качество; для полного сравнения приложен результат 09.


In [14]:
# --- baseline работы 09 (LLaMA-Factory): берётся только из реальных артефактов 09; иначе явно помечается как отсутствующий
baseline = {'available': False}
p9 = find_lab9('predictions.jsonl')
if p9 is not None:
    rows9 = load_jsonl(p9); b9 = {}
    for s in EVAL_SPLITS:
        sel = [r for r in rows9 if r['split'] == s]
        if not sel: continue
        e = evaluate_answers([{'id': r['id'], 'split': s, 'gold': r['gold'], 'response': r['sft_response']} for r in sel], s)     # та же метрика, что в 10
        b9[s] = {k: e[k] for k in ('n', 'exact_match', 'format_rate')}
        assert {r['id'] for r in sel} == ids['r8_a16'] & {r['id'] for r in by_split[s]}, 'наборы ID 09 и 10 различаются'
    baseline = {'available': bool(b9), 'source': str(p9), 'metric_note': 'метрика 10 пересчитана по sft_response из predictions.jsonl работы 09', 'splits': b9}
if not baseline['available']:
    baseline['reason'] = 'артефакты работы 09 (predictions.jsonl) не найдены: baseline LLaMA-Factory отсутствует (это НЕ нулевое качество); задайте CV_LAB9_DIR'

# --- сопоставимость скорости: заявляется только при совпадении GPU, шагов, effective batch и обучаемых модулей
v9, ts9 = find_lab9('version.json'), find_lab9('train_summary.json')
checks = {'same_gpu': bool(v9) and json.loads(v9.read_text()).get('gpu') == GPU,
          'same_steps': bool(ts9) and json.loads(ts9.read_text()).get('global_step') == MAX_STEPS,
          'same_effective_batch': False, 'same_trainable_modules': False, 'wall_clock_of_09_recorded': False}
speed = {'comparable': all(checks.values()), 'checks': checks,
         'claimed_speedup': None, 'unsloth_seconds': {t: bench[t]['total_seconds'] for t in RUNS},
         'reason': 'в артефактах 09 не записаны wall-clock, effective batch и набор обучаемых модулей в сопоставимом виде; ускорение не заявляется'}
comparison = {'ranks': {'quality': quality, 'paired_r16_vs_r8': paired, 'resources': {t: {k: bench[t][k] for k in ('total_seconds', 'peak_cuda_memory_gb', 'tokens_per_second', 'n_trainable_params', 'loss_last')} for t in RUNS},
                        'ratio_r16_over_r8': bench['ratio_r16_over_r8']},
              'baseline_lab09_llamafactory': baseline, 'speed_comparison': speed}
concl = []
for s in EVAL_SPLITS:
    p = paired[s]; lo, hi = p['bootstrap_95ci']
    concl.append(f"{s}: EM r8={quality['r8_a16'][s]['exact_match']:.3f}, r16={quality['r16_a16'][s]['exact_match']:.3f}; Δ(r16−r8)={p['delta_em_r16_minus_r8']:+.3f}, 95% CI [{lo:+.3f}; {hi:+.3f}] - "
                 + ('различие не отличимо от нуля' if lo <= 0 <= hi else ('r16 лучше' if lo > 0 else 'r16 хуже')))
r = bench['ratio_r16_over_r8']
concl.append(f"ресурсы: r16 требует {r['time']:.2f}× времени и {r['peak_memory']:.2f}× пиковой памяти r8 при {r['trainable_params']:.2f}× параметров; Δloss(last10)={r['delta_loss_last']:+.4f}")
concl.append('baseline 09: ' + ('сопоставлен по тем же ID' if baseline['available'] else 'отсутствует (не принимается за нулевое качество)'))
concl.append('скорость Unsloth против LLaMA-Factory не сравнивается: ' + speed['reason'])
comparison['conclusion'] = concl
save_json('comparison.json', comparison); print('\n'.join(concl))

id_test: EM r8=0.922, r16=0.906; Δ(r16−r8)=-0.016, 95% CI [-0.047; +0.000] - различие не отличимо от нуля
ood_test: EM r8=0.734, r16=0.734; Δ(r16−r8)=+0.000, 95% CI [-0.047; +0.047] - различие не отличимо от нуля
ресурсы: r16 требует 0.96× времени и 1.05× пиковой памяти r8 при 2.00× параметров; Δloss(last10)=-0.0111
baseline 09: отсутствует (не принимается за нулевое качество)
скорость Unsloth против LLaMA-Factory не сравнивается: в артефактах 09 не записаны wall-clock, effective batch и набор обучаемых модулей в сопоставимом виде; ускорение не заявляется


### Творческое мини-исследование

*Бонус: до 1 балла сверх 10.*

**Постановка.** Сформулируйте проверяемую гипотезу о предметном ограничении метода данной лабораторной и проверьте её на дополнительных реальных входах без использования отложенного теста для выбора гипотезы. Сопоставьте результаты с основным экспериментом и объясните расхождения.

**Результат.** Отдельный файл artifacts/bonus.md с гипотезой, протоколом, наблюдениями и выводом.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена по сохранённым предсказаниям или изображениям и приведено воспроизводимое сравнение.

In [15]:
# Гипотеза (до запуска; отложенный test не используется): больший ранг снижает train-loss (ёмкость), но не улучшает качество на неиспользованных
# данных: Δ EM(r16−r8) на validation (64 доп. реальных изображения) не отличима от 0 (95% bootstrap CI содержит 0), хотя Δloss < 0.
VAL = by_split['validation']; bonus_rows = {}
for tag in RUNS:
    path = ARTIFACTS / f'bonus_validation_{tag}.jsonl'; predict_all(tag, VAL, path, sanity=False); bonus_rows[tag] = load_jsonl(path)
ev = {tag: evaluate_answers(rows, 'validation') for tag, rows in bonus_rows.items()}
c = {tag: {p['id']: p['correct'] for p in e['per_example']} for tag, e in ev.items()}
keys = sorted(c['r8_a16']); d = np.array([c['r16_a16'][k] - c['r8_a16'][k] for k in keys]); br = np.random.RandomState(SEED + 3)
boots = [d[br.randint(0, len(d), len(d))].mean() for _ in range(2000)]; lo, hi = np.percentile(boots, [2.5, 97.5])
dloss = bench['ratio_r16_over_r8']['delta_loss_last']
h_loss, h_em = dloss < 0, lo <= 0 <= hi
md = f"""# Бонус: ёмкость адаптера против обобщения

## Гипотеза
Rank 16 снижает train-loss относительно rank 8, но не улучшает exact match на неиспользованных данных: доверительный интервал разности EM(r16−r8) содержит 0.

## Протокол
- Дополнительные реальные входы: split validation ({len(VAL)} изображений ScienceQA), не входящий в train и в test; отложенный test для выбора и проверки гипотезы не использовался.
- Те же адаптеры, общий `infer` (greedy, max_new_tokens={MAX_NEW_TOKENS}) и та же строгая метрика; paired bootstrap по ID (2000 resamples, seed {SEED + 3}).

## Наблюдения
- Δ train-loss (последние 10 шагов, r16−r8): {dloss:+.4f}.
- EM на validation: r8 = {ev['r8_a16']['exact_match']:.3f}, r16 = {ev['r16_a16']['exact_match']:.3f}; Δ = {d.mean():+.3f}, 95% CI [{lo:+.3f}; {hi:+.3f}], format rate: {ev['r8_a16']['format_rate']:.3f} / {ev['r16_a16']['format_rate']:.3f}.
- Основной эксперимент (ID/OOD test): {'; '.join(f"{s}: Δ={paired[s]['delta_em_r16_minus_r8']:+.3f} CI [{paired[s]['bootstrap_95ci'][0]:+.3f}; {paired[s]['bootstrap_95ci'][1]:+.3f}]" for s in EVAL_SPLITS)}.

## Вывод
Часть про loss {'подтверждена' if h_loss else 'НЕ подтверждена'}, часть про EM {'подтверждена (различие не отличимо от нуля)' if h_em else 'НЕ подтверждена (интервал не содержит 0)'}: гипотеза {'подтверждена' if (h_loss and h_em) else 'не подтверждена полностью'}.
Ограничения: n = {len(VAL)}, один seed на ранг, loss обучающий, разные ранги не различаются по бюджету шагов; расхождения с test объясняются составом изображений и шумом малой выборки.
"""
(ARTIFACTS / 'bonus.md').write_text(md); save_json('bonus_stats.json', {'delta_em': float(d.mean()), 'ci': [float(lo), float(hi)], 'delta_loss': dloss}); print(md)

==((====))==  Unsloth 2026.10.6: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

==((====))==  Unsloth 2026.10.6: Fast Qwen2_5_Vl patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.5.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.30.mlp.down_proj: no quant_state found


Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

# Бонус: ёмкость адаптера против обобщения

## Гипотеза
Rank 16 снижает train-loss относительно rank 8, но не улучшает exact match на неиспользованных данных: доверительный интервал разности EM(r16−r8) содержит 0.

## Протокол
- Дополнительные реальные входы: split validation (64 изображений ScienceQA), не входящий в train и в test; отложенный test для выбора и проверки гипотезы не использовался.
- Те же адаптеры, общий `infer` (greedy, max_new_tokens=8) и та же строгая метрика; paired bootstrap по ID (2000 resamples, seed 45).

## Наблюдения
- Δ train-loss (последние 10 шагов, r16−r8): -0.0111.
- EM на validation: r8 = 0.859, r16 = 0.859; Δ = +0.000, 95% CI [-0.062; +0.062], format rate: 1.000 / 1.000.
- Основной эксперимент (ID/OOD test): id_test: Δ=-0.016 CI [-0.047; +0.000]; ood_test: Δ=+0.000 CI [-0.047; +0.047].

## Вывод
Часть про loss подтверждена, часть про EM подтверждена (различие не отличимо от нуля): гипотеза подтверждена.
Ограничения: n = 64, один seed на ранг, loss обуча

## Анализ результатов

Сопоставьте измеренные показатели на фиксированной отложенной выборке, укажите ограничения выборки и разберите ошибки по сохранённым предсказаниям. FAST_DEV_RUN не является основанием для итогового вывода.

## Требования к сдаче

Запустите полный режим, приложите конфигурацию, версии зависимостей, артефакты и инструкции повторного запуска. Не сдавайте фиктивные измерения.

## Инструкция повторного запуска

1. Colab, GPU T4 (или выше). Сначала `%pip install -q unsloth` (Unsloth подбирает совместимые `transformers`, `trl`, `peft`), затем `%pip install -q datasets scikit-learn matplotlib`. Версии фиксируются в `artifacts/protocol.json`.
2. Положите рядом артефакты работы 09 (`CV_LAB9_DIR=/путь/к/artifacts`) — тогда манифест будет сверен по ID/хешам, а baseline LLaMA-Factory пересчитан той же строгой метрикой. Без них baseline явно помечается отсутствующим.
3. `FAST_DEV_RUN=True` — 2 optimizer step × 2 микробатча на smoke-манифесте (проверка стека, не результат); `False` — 100 шагов × 16 микробатчей для каждого ранга.
4. Прогоны возобновляемы на уровне ранга (готовый `training.json` + адаптер пропускаются), инференс дописывает `predictions_*.jsonl` построчно.
5. Имя каталога `unsloth_r16_a16` дано по заданию, фактический alpha для rank 16 равен 32 (по формулировке «меняется только ранг и соответствующий alpha»), это записано в `protocol.json` и `training.json`.

In [16]:
save_json('environment.json', {'config': {'SEED': SEED, 'FAST_DEV_RUN': FAST_DEV_RUN, 'MODEL_ID': MODEL_ID, 'MAX_STEPS': MAX_STEPS, 'ACCUM': ACCUM, 'LR': LR,
                                          'IMAGE_MAX_PIXELS': IMAGE_MAX_PIXELS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'SCIENCEQA_REVISION': SCIENCEQA_REVISION},
                               'protocol': protocol})
print(sorted(p.name for p in ARTIFACTS.iterdir()))

['ATTRIBUTION.md', 'benchmark.json', 'bonus.md', 'bonus_stats.json', 'bonus_validation_r16_a16.jsonl', 'bonus_validation_r8_a16.jsonl', 'comparison.json', 'environment.json', 'loss_comparison.png', 'manifest.jsonl', 'manifest_check.json', 'predictions_r16_a16.jsonl', 'predictions_r8_a16.jsonl', 'protocol.json', 'quality.json', 'unsloth_r16_a16', 'unsloth_r8_a16']
